# NB00b — LC-MS Feature Detection with pyopenms
**Replaces:** greedy m/z ± 0.01 Da + RT ± 30 s binning in NB00 cells 6–7  
**Run once per cohort** when raw `.mzML` files are available.

## Pipeline
```
*.mzML (per sample)
  └─ FeatureFinderMetabo       — mass trace → isotope cluster → metabolite features
       └─ MetaboliteFeatureDeconvolution  — group [M+H]+, [M+Na]+, [M+K]+ into neutral mass
            └─ MapAlignerPoseClustering   — RT alignment across samples
                 └─ FeatureGroupingAlgorithmQT — link features across samples
                      └─ KEGG exact-mass lookup → mtb.tsv (same format as NB01 input)
```

## Requirements
```
pip install pyopenms requests
```
Place raw `.mzML` files in `Data/mzML/{DATASET}/`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import pyopenms as oms
    print(f"pyopenms {oms.__version__} loaded")
except ImportError:
    raise ImportError(
        "pyopenms is required for this notebook.\n"
        "Install with: pip install pyopenms"
    )

# ── Configuration ─────────────────────────────────────────────────────────────
# Change DATASET to match the cohort being processed
DATASET     = "YACHIDA-CRC-2019"

DATA_DIR    = Path(r"C:\Users\andna\Documents\KI\Data")
MZML_DIR    = DATA_DIR / "mzML" / DATASET
MTB_OUT_DIR = DATA_DIR / "MetabolomicsOut" / DATASET
MTB_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Feature detection parameters
MASS_ERROR_PPM    = 5.0          # Orbitrap: 5 ppm; ToF/Q-ToF: 10–15 ppm
NOISE_THRESHOLD   = 1000.0       # minimum peak intensity
CHARGE_MAX        = 3            # most metabolites: z = 1–3
RT_WINDOW_SEC     = 30.0         # ± 30 s for feature linking across samples
GLASSO_THRESHOLD  = 10.0         # m/z tolerance in ppm for feature grouping

mzml_files = sorted(MZML_DIR.glob("*.mzML"))
print(f"Found {len(mzml_files)} mzML files in {MZML_DIR}")
if not mzml_files:
    print(f"Place raw .mzML files in {MZML_DIR} to proceed.")

## Step 1 — Per-sample peak picking with FeatureFinderMetabo

In [ ]:
def run_feature_finder(mzml_path: Path, out_featurexml: Path) -> int:
    """
    Pick chromatographic peaks and group isotope clusters into metabolite features.
    Returns the number of features detected.
    """
    # Load spectrum
    exp = oms.MSExperiment()
    oms.MzMLFile().load(str(mzml_path), exp)
    exp.sortSpectra(True)

    # 1. Mass trace detection
    mtd        = oms.MassTraceDetection()
    mtd_params = mtd.getParameters()
    mtd_params.setValue("mass_error_ppm",    MASS_ERROR_PPM)
    mtd_params.setValue("noise_threshold_int", NOISE_THRESHOLD)
    mtd_params.setValue("min_trace_length",  3.0)   # seconds
    mtd_params.setValue("max_trace_length",  300.0) # seconds
    mtd.setParameters(mtd_params)

    mass_traces: list = []
    mtd.run(exp, mass_traces, 0)

    # 2. Elution peak detection (split traces at valley points)
    epd               = oms.ElutionPeakDetection()
    mass_traces_split: list = []
    epd.detectPeaks(mass_traces, mass_traces_split)

    # 3. Isotope grouping → metabolite features
    ff_metabo  = oms.FeatureFindingMetabo()
    ff_params  = ff_metabo.getParameters()
    ff_params.setValue("isotope_filtering_model", "metabolites (5% RMS)")
    ff_params.setValue("charge_lower_bound",      1)
    ff_params.setValue("charge_upper_bound",      CHARGE_MAX)
    ff_params.setValue("report_summed_ints",      "true")
    ff_metabo.setParameters(ff_params)

    feature_map = oms.FeatureMap()
    ff_metabo.run(mass_traces_split, feature_map, [])
    feature_map.setPrimaryMSRunPath([str(mzml_path).encode()])

    oms.FeatureXMLFile().store(str(out_featurexml), feature_map)
    n = feature_map.size()
    print(f"  {mzml_path.name}: {n:,} features")
    return n


# Run per-sample (skip if already done)
feature_counts = {}
for mzml_file in mzml_files:
    out_xml = MTB_OUT_DIR / (mzml_file.stem + ".featureXML")
    if out_xml.exists():
        print(f"  {mzml_file.name}: already processed — skipping")
        fm = oms.FeatureMap()
        oms.FeatureXMLFile().load(str(out_xml), fm)
        feature_counts[mzml_file.stem] = fm.size()
    else:
        feature_counts[mzml_file.stem] = run_feature_finder(mzml_file, out_xml)

print(f"\nMedian features per sample: {int(np.median(list(feature_counts.values()))):,}")

## Step 2 — Adduct deconvolution with MetaboliteFeatureDeconvolution
Groups `[M+H]+`, `[M+Na]+`, `[M+K]+`, `[M+NH4]+` etc. into a single neutral-mass feature per metabolite.

In [ ]:
def deconvolve_adducts(featurexml_path: Path, out_path: Path) -> None:
    feature_map = oms.FeatureMap()
    oms.FeatureXMLFile().load(str(featurexml_path), feature_map)

    mfd        = oms.MetaboliteFeatureDeconvolution()
    mfd_params = mfd.getParameters()
    # Adduct probability weights (must sum to ~1.0 per charge state)
    mfd_params.setValue(
        "potential_adducts",
        b"H:+:0.40 Na:+:0.20 K:+:0.10 NH4:+:0.10 H-1O-1:+:0.10 H-2O-2:+:0.05 H-1:+:0.05",
    )
    mfd_params.setValue("charge_min",        1)
    mfd_params.setValue("charge_max",        CHARGE_MAX)
    mfd_params.setValue("mass_max_diff",     10.0)  # ppm
    mfd_params.setValue("unit",              b"ppm")
    mfd.setParameters(mfd_params)

    feature_map_out = oms.FeatureMap()
    cons_map        = oms.ConsensusMap()
    mfd.compute(feature_map, feature_map_out, cons_map)
    oms.FeatureXMLFile().store(str(out_path), feature_map_out)


for featurexml in sorted(MTB_OUT_DIR.glob("*.featureXML")):
    if "_deconv" in featurexml.name:
        continue
    out_path = MTB_OUT_DIR / (featurexml.stem + "_deconv.featureXML")
    if not out_path.exists():
        deconvolve_adducts(featurexml, out_path)
        print(f"  Deconvolved: {featurexml.name}")
    else:
        print(f"  {featurexml.name}: deconv already exists — skipping")

print("Adduct deconvolution complete.")

## Step 3 — Retention-time alignment with MapAlignerPoseClustering
Corrects systematic RT drift across samples using a pose-clustering algorithm.

In [ ]:
deconv_files = sorted(MTB_OUT_DIR.glob("*_deconv.featureXML"))
print(f"Aligning {len(deconv_files)} feature maps ...")

# Load all deconvolved feature maps
feature_maps = []
for f in deconv_files:
    fm = oms.FeatureMap()
    oms.FeatureXMLFile().load(str(f), fm)
    fm.setPrimaryMSRunPath([str(f).encode()])
    feature_maps.append(fm)

# Configure aligner
aligner     = oms.MapAlignerPoseClustering()
alg_params  = aligner.getParameters()
alg_params.setValue("max_num_peaks_considered",        1000)
alg_params.setValue("pairfinder:distance_MZ:max_difference", GLASSO_THRESHOLD)
alg_params.setValue("pairfinder:distance_MZ:unit",     b"ppm")
alg_params.setValue("pairfinder:distance_RT:max_difference", RT_WINDOW_SEC * 2)
aligner.setParameters(alg_params)

transformations: list = []
aligner.align(feature_maps, transformations)

# Apply RT transformations and save
transformer = oms.MapAlignmentTransformer()
for fm, tf, f in zip(feature_maps, transformations, deconv_files):
    transformer.transformRetentionTimes(fm, tf, True)
    out_path = MTB_OUT_DIR / (f.stem + "_aligned.featureXML")
    oms.FeatureXMLFile().store(str(out_path), fm)

print(f"RT alignment complete. {len(feature_maps)} maps aligned.")

## Step 4 — Feature linking and consensus map
Groups corresponding features across samples into a consensus, filling missing values with 0.

In [ ]:
aligned_files = sorted(MTB_OUT_DIR.glob("*_aligned.featureXML"))
aligned_maps  = []
for f in aligned_files:
    fm = oms.FeatureMap()
    oms.FeatureXMLFile().load(str(f), fm)
    fm.setPrimaryMSRunPath([str(f).encode()])
    aligned_maps.append(fm)

# Feature grouping: QT-clique algorithm (handles missing features gracefully)
feature_grouper = oms.FeatureGroupingAlgorithmQT()
fg_params       = feature_grouper.getParameters()
fg_params.setValue("distance_MZ:max_difference", GLASSO_THRESHOLD)
fg_params.setValue("distance_MZ:unit",           b"ppm")
fg_params.setValue("distance_RT:max_difference", RT_WINDOW_SEC)
fg_params.setValue("use_identifications",        b"false")
feature_grouper.setParameters(fg_params)

consensus_map = oms.ConsensusMap()
feature_grouper.group(aligned_maps, consensus_map)
oms.ConsensusXMLFile().store(str(MTB_OUT_DIR / "consensus.consensusXML"), consensus_map)
print(f"Consensus map: {consensus_map.size()} feature groups across {len(aligned_maps)} samples")

## Step 5 — KEGG annotation and export to mtb.tsv

In [ ]:
import requests, time

def kegg_exact_mass_lookup(mz: float, tol_da: float = 0.005) -> str:
    """Query KEGG REST API for compound by exact mass. Returns KEGG ID or mz_* fallback."""
    try:
        url = f"https://rest.kegg.jp/find/compound/{mz:.4f}/exact_mass"
        r   = requests.get(url, timeout=10)
        if r.ok and r.text.strip():
            # First hit; format: 'cpd:C00134\tPutrescine\n'
            first_line = r.text.strip().split("\n")[0]
            kegg_id    = first_line.split("\t")[0].replace("cpd:", "")
            name       = first_line.split("\t")[1] if "\t" in first_line else kegg_id
            return f"{kegg_id}_{name.split(';')[0].strip()}"
    except Exception:
        pass
    return f"mz_{mz:.4f}"


# Extract sample names from consensus map column headers
col_headers  = consensus_map.getColumnHeaders()
sample_names = []
for h in col_headers:
    fname = Path(h.filename.decode() if isinstance(h.filename, bytes) else h.filename)
    # strip suffixes added during processing
    stem = fname.stem
    for suffix in ("_deconv_aligned", "_deconv", "_aligned"):
        stem = stem.replace(suffix, "")
    sample_names.append(stem)

# Build intensity matrix
rows = []
for feature in consensus_map:
    mz = feature.getMZ()
    rt = feature.getRT()
    intensities = {s: 0.0 for s in sample_names}
    for handle in feature.getFeatureList():
        idx = handle.getMapIndex()
        if idx < len(sample_names):
            intensities[sample_names[idx]] = handle.getIntensity()
    rows.append({"mz": mz, "rt_sec": rt, **intensities})

intensity_df = pd.DataFrame(rows)
print(f"Raw consensus: {len(intensity_df)} features × {len(sample_names)} samples")

# KEGG annotation (batched with 0.2 s delay to respect KEGG rate limit)
print("Querying KEGG REST API (may take several minutes) ...")
kegg_ids = []
for i, mz in enumerate(intensity_df["mz"]):
    kegg_ids.append(kegg_exact_mass_lookup(mz))
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(intensity_df)} annotated")
        time.sleep(0.2)

intensity_df["kegg_id"] = kegg_ids

# Pivot: samples (rows) × KEGG IDs (cols), log10(intensity + 1) transform
mtb_matrix = (
    intensity_df
    .set_index("kegg_id")
    .drop(columns=["mz", "rt_sec"])
    .T
    .apply(lambda col: np.log10(col.astype(float) + 1))
)
mtb_matrix.index.name = "sample_id"

# Drop features with zero variance (uniformly undetected)
n_before = mtb_matrix.shape[1]
mtb_matrix = mtb_matrix.loc[:, mtb_matrix.var(axis=0) > 0]
print(f"Features after zero-variance filter: {n_before} → {mtb_matrix.shape[1]}")

# Save in same format as existing {DATASET} mtb.tsv files
out_tsv = DATA_DIR / f"{DATASET} mtb.tsv"
mtb_matrix.to_csv(out_tsv, sep="\t")
print(f"\nSaved: {out_tsv}")
print(f"  Shape: {mtb_matrix.shape[0]} samples × {mtb_matrix.shape[1]} features")
print(f"  KEGG-annotated: {sum(not k.startswith('mz_') for k in mtb_matrix.columns)} / {mtb_matrix.shape[1]}")

# Also save metabolite map file (KEGG ID → name)
map_rows = []
for kid in mtb_matrix.columns:
    if "_" in kid and not kid.startswith("mz_"):
        kegg, name = kid.split("_", 1)
    else:
        kegg, name = kid, kid
    map_rows.append({"kegg_id": kegg, "name": name, "full_id": kid})
map_df = pd.DataFrame(map_rows)
map_out = DATA_DIR / f"{DATASET} mtb.map.tsv"
map_df.to_csv(map_out, sep="\t", index=False)
print(f"Saved metabolite map: {map_out}")